# Retinotopy stimulus recovery

For $N$ subjects with HCP 7T retinotopy fMRI, we jointly factorise the stimulus
aperture $X_t$ (HRF-convolved, pixels x time) and each subject's V1 fMRI
$C_{s,t}$ (vertices x time) across the five training tasks, with a shared
per-task temporal factor:

$$X_t = A_{\text{stim}} S_t^T \quad\text{and}\quad C_{s,t} = A_s S_t^T.$$

RETCCW is held out. At test time, we solve $S_{\text{CCW}}$ from held-out
$C_{s,\text{CCW}}$ via ridge regression with $A_s$, then reconstruct the
stimulus as $\hat X_{\text{CCW}} = A_{\text{stim}} S_{\text{CCW}}^T$, averaged
across subjects.

In [ ]:
import os, sys, pickle
from pathlib import Path
import numpy as np
import h5py
import nibabel as nib
from scipy.io import loadmat
from scipy.stats import gamma, pearsonr
from scipy.signal import fftconvolve
from scipy.linalg import lstsq
from sklearn.metrics import r2_score

sys.path.insert(0, os.path.abspath('..'))
from pathfinder.decomp import JointOuterDecomp

DATA_ROOT = Path('/well/win-biobank/users/gbb787/analyzePRF_HCP7TRET')
APERTURES_DIR = Path('apertures')
SUBJECT_IDS = ['999997']

TASKS = ['RETBAR1', 'RETBAR2', 'RETCCW', 'RETCW', 'RETCON', 'RETEXP']
PHASES = {'RETBAR1': 'AP', 'RETBAR2': 'PA', 'RETCCW': 'AP',
          'RETCW': 'PA', 'RETCON': 'PA', 'RETEXP': 'AP'}
APERTURE_FILES = {t: f'RET{t[3:]}small.mat' for t in TASKS}
HELD_OUT_TASK = 'RETCCW'

N_COMPONENTS = 20
N_ITER = 100
ALPHA_REG = 0.1
BATCH_SIZE = 150

## HRF convolution

Canonical double-gamma HRF sampled at TR, normalised to unit area.

In [ ]:
def create_hrf(tr=1.0, oversampling=16, duration=32.0):
    dt = tr / oversampling
    t = np.arange(0, duration, dt)
    hrf = gamma.pdf(t, 5) - 0.35 * gamma.pdf(t, 12)
    hrf = hrf[::oversampling]
    return hrf / hrf.sum()


def convolve_with_hrf(stimulus, tr=1.0):
    hrf = create_hrf(tr=tr)
    n_time = stimulus.shape[1]
    conv = fftconvolve(stimulus.T, hrf[:, None], mode='full', axes=0)
    return conv[:n_time, :].T

## Load stimuli and fMRI

Apertures are reshaped to (pixels x time) and HRF-convolved. fMRI is
demeaned and restricted to V1 vertices with PRF $R^2 > 55$.

In [ ]:
def load_aperture(task):
    path = APERTURES_DIR / APERTURE_FILES[task]
    try:
        ap = loadmat(str(path))['stim']
    except NotImplementedError:
        with h5py.File(str(path), 'r') as f:
            ap = f['stim'][()]
    # HCP format is (time, height, width) -> (pixels, time)
    return ap.transpose(1, 2, 0).reshape(-1, ap.shape[0])


v1_mask = nib.load('V1.dscalar.nii').get_fdata()
r2_map = nib.load('GroupAvg.R2.dscalar.nii').get_fdata()
vertex_idx = np.where((v1_mask != 0) * (r2_map > 55))[1]
print(f'V1 vertices with R^2>55: {vertex_idx.size}')


def load_fmri(sid, task):
    fname = f'tfMRI_RET{task[3:]}_7T_{PHASES[task]}_Atlas_MSMAll_hp2000_clean.dtseries.nii'
    data = nib.load(str(DATA_ROOT / sid / fname)).get_fdata().T  # (vertices, time)
    data = data - data.mean(axis=1, keepdims=True)
    return data[vertex_idx, :]


fmri = {sid: {t: load_fmri(sid, t) for t in TASKS} for sid in SUBJECT_IDS}
n_tr = fmri[SUBJECT_IDS[0]][TASKS[0]].shape[1]

stimulus = {}
for t in TASKS:
    ap = load_aperture(t)
    if ap.shape[1] > n_tr:
        ap = ap[:, :n_tr]
    elif ap.shape[1] < n_tr:
        ap = np.hstack([ap, np.zeros((ap.shape[0], n_tr - ap.shape[1]))])
    stimulus[t] = convolve_with_hrf(ap)
print(f'Stimulus shape per task: {stimulus[TASKS[0]].shape}')

## Build the joint factorisation

For each training task $t$ (all five except RETCCW):

* stimulus uses shared spatial index $\alpha = 0$,
* each subject's fMRI uses a subject-specific $\alpha = 1 + s$,
* all matrices for task $t$ share the same temporal index $\beta = t$.

In [ ]:
train_tasks = [t for t in TASKS if t != HELD_OUT_TASK]
Clist, alpha_map, beta_map = [], [], []
for bi, task in enumerate(train_tasks):
    Clist.append(stimulus[task]); alpha_map.append(0); beta_map.append(bi)
    for si, sid in enumerate(SUBJECT_IDS):
        Clist.append(fmri[sid][task]); alpha_map.append(1 + si); beta_map.append(bi)

print(f'{len(Clist)} matrices, {max(alpha_map)+1} spatial modes '
      f'(1 stim + {len(SUBJECT_IDS)} fMRI), {max(beta_map)+1} temporal modes')

## Fit PathFinder

In [ ]:
model = JointOuterDecomp(
    n_components=N_COMPONENTS, n_iter=N_ITER, alpha=ALPHA_REG,
    batch_size=BATCH_SIZE, verbose=True,
)
model.fit(Clist, alpha=alpha_map, beta=beta_map)
print(f'Final loss: {np.mean(model._loss[-1]):.4f}')

## Reconstruct held-out stimulus

For each subject, solve $S = (A_s^T A_s + \alpha I)^{-1} A_s^T C_{s,\text{CCW}}$
and reconstruct $\hat X_s = A_{\text{stim}} S^T$. Average across subjects.

In [ ]:
A_stim = model._A[0]
I = np.eye(N_COMPONENTS)

subject_preds = []
for si, sid in enumerate(SUBJECT_IDS):
    A_s = model._A[1 + si]
    C_held = fmri[sid][HELD_OUT_TASK]
    S = lstsq(A_s.T @ A_s + ALPHA_REG * I, A_s.T @ C_held)[0].T
    subject_preds.append(A_stim @ S.T)

stim_pred = np.mean(subject_preds, axis=0)
stim_true = stimulus[HELD_OUT_TASK]

overall_r, _ = pearsonr(stim_true.ravel(), stim_pred.ravel())
overall_r2 = r2_score(stim_true.ravel(), stim_pred.ravel())
spatial_r = np.mean([pearsonr(stim_true[:, t], stim_pred[:, t])[0]
                     for t in range(stim_true.shape[1])
                     if stim_true[:, t].std() > 1e-10])
print(f'{HELD_OUT_TASK}: r={overall_r:.3f}, R^2={overall_r2:.3f}, '
      f'mean per-frame spatial r={spatial_r:.3f}')

## Save

In [ ]:
OUT_DIR = Path('retinotopy_recovery')
OUT_DIR.mkdir(exist_ok=True, parents=True)
with open(OUT_DIR / f'{HELD_OUT_TASK}_recovery.pkl', 'wb') as f:
    pickle.dump({
        'true_stimulus': stim_true,
        'predicted_stimulus': stim_pred,
        'predictions_per_subject': subject_preds,
        'subject_ids': SUBJECT_IDS,
        'metrics': {'r': overall_r, 'r2': overall_r2, 'spatial_r': spatial_r},
        'loss': np.array(model._loss),
    }, f)
print(f'Saved to {OUT_DIR}/{HELD_OUT_TASK}_recovery.pkl')